# DSPy Hands-On (Google Colab)

**Program, don't prompt.** DSPy lets you build LLM apps as modular Python programs: declare inputs/outputs (`Signature`), choose a strategy (`Module`), score quality (`metric`), then auto-improve prompts/demos (`Optimizer`).

### What you'll build
1. Configure an LM (OpenAI or Gemini)
2. Signatures → `Predict` / `ChainOfThought`
3. A small multi-step custom `Module` (classify → answer)
4. A metric + `BootstrapFewShot` optimization (before/after)
5. Save / reload the optimized program

### Cost tip
Use a cheap model (`gpt-4o-mini` or `gemini-2.0-flash`). Optimization makes several LM calls — keep the trainset tiny in this notebook.

Docs: [dspy.ai](https://dspy.ai/)

## 0) Install

In [31]:
%pip install -q -U dspy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1) Set your OpenAI API key

Paste your key below (or leave blank and use Colab Secrets / shell env).

In [33]:
import os

# Paste your key here (do not commit real keys)
os.environ["OPENAI_API_KEY"] = ""  # e.g. "sk-..."

# Optional Gemini fallback
# os.environ["GOOGLE_API_KEY"] = ""

# assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY in the cell above"

## 2) Configure the language model

Uses `OPENAI_API_KEY` from the env cell above (or Colab Secrets / shell env).
For Gemini, set `PROVIDER = "gemini"` and `GOOGLE_API_KEY`.

In [20]:
from getpass import getpass

import dspy

# "openai" or "gemini"
PROVIDER = "openai"

MODELS = {
    "openai": "openai/gpt-4o-mini",
    "gemini": "gemini/gemini-2.0-flash",
}


def get_secret(name: str) -> str:

    return os.environ.get(name)


if PROVIDER == "openai":
    api_key = get_secret("OPENAI_API_KEY") or getpass("OPENAI_API_KEY: ")
    os.environ["OPENAI_API_KEY"] = api_key
    lm = dspy.LM(MODELS["openai"], api_key=api_key)
elif PROVIDER == "gemini":
    api_key = (
        get_secret("GOOGLE_API_KEY")
        or get_secret("GEMINI_API_KEY")
        or getpass("GOOGLE_API_KEY: ")
    )
    os.environ["GOOGLE_API_KEY"] = api_key
    lm = dspy.LM(MODELS["gemini"], api_key=api_key)
else:
    raise ValueError("PROVIDER must be 'openai' or 'gemini'")

dspy.configure(lm=lm)
print("configured:", MODELS[PROVIDER])
print(lm(messages=[{"role": "user", "content": "Reply with exactly: ok"}])[0])

configured: openai/gpt-4o-mini
ok


## 3) Your first DSPy program: `Predict` + string signature

A **signature** is `inputs -> outputs`. Field names matter — the LM reads them.

In [21]:
qa = dspy.Predict("question -> answer")
pred = qa(question="In one sentence: what is DSPy?")
print(pred.answer)

DSPy is a programming framework designed for building and deploying machine learning models using a declarative approach that emphasizes simplicity and interpretability.


In [22]:
# See the prompt DSPy actually sent
dspy.inspect_history(n=1)





[2026-08-10T22:37:56.728808]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer` (str):
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Given the fields `question`, produce the fields `answer`.


User message:

[[ ## question ## ]]
In one sentence: what is DSPy?

Respond with the corresponding output fields, starting with the field `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## answer ## ]]
DSPy is a programming framework designed for building and deploying machine learning models using a declarative approach that emphasizes simplicity and interpretability.

[[ ## completed ## ]]







## 4) Class-based signatures (instructions + typed fields)

Use a class when you want a docstring (task instruction), field descriptions, or richer types.

In [23]:
from typing import Literal


class TicketTriage(dspy.Signature):
    """Classify a support ticket and suggest the next action."""

    ticket: str = dspy.InputField(desc="Raw customer message")
    category: Literal["billing", "bug", "howto", "other"] = dspy.OutputField()
    priority: Literal["low", "medium", "high"] = dspy.OutputField()
    next_action: str = dspy.OutputField(desc="One concrete next step for the agent")


triage = dspy.Predict(TicketTriage)
out = triage(ticket="I was charged twice for the Pro plan this month and need a refund today.")
print(out)

Prediction(
    category='billing',
    priority='high',
    next_action='Process the refund for the duplicate charge immediately.'
)


## 5) Swap strategies without rewriting the task

Same signature, different module:
- `Predict` — direct mapping
- `ChainOfThought` — asks for reasoning first, then outputs

In [24]:
cot_triage = dspy.ChainOfThought(TicketTriage)
out = cot_triage(ticket="App crashes when I export a PDF on iOS 18.")
print("reasoning:", out.reasoning)
print("category:", out.category)
print("priority:", out.priority)
print("next_action:", out.next_action)

reasoning: The customer is experiencing a crash in the application when attempting to export a PDF on iOS 18, which indicates a potential bug in the software that needs to be addressed.
category: bug
priority: high
next_action: Escalate the issue to the development team for further investigation and resolution.


## 6) Compose a custom `dspy.Module`

Multi-step programs are plain Python: name each stage, call them in `forward()`.

In [25]:
class ClassifyIntent(dspy.Signature):
    """Label the user's intent for a tech FAQ bot."""

    question: str = dspy.InputField()
    intent: Literal["definition", "howto", "comparison", "other"] = dspy.OutputField()


class AnswerQuestion(dspy.Signature):
    """Answer briefly and accurately for the given intent."""

    question: str = dspy.InputField()
    intent: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="2-4 sentences max")


class FAQBot(dspy.Module):
    def __init__(self):
        super().__init__()
        self.classify = dspy.Predict(ClassifyIntent)
        self.answer = dspy.ChainOfThought(AnswerQuestion)

    def forward(self, question: str):
        intent = self.classify(question=question).intent
        return self.answer(question=question, intent=intent)


bot = FAQBot()
result = bot(question="What is the difference between fine-tuning and RAG?")
print("reasoning:", result.reasoning)
print("answer:", result.answer)

reasoning: Fine-tuning involves adjusting a pre-trained model on a specific dataset to improve its performance on a particular task, allowing it to learn task-specific features. In contrast, Retrieval-Augmented Generation (RAG) combines generative models with retrieval systems, where the model retrieves relevant information from a knowledge base to enhance its responses, rather than solely relying on learned parameters.
answer: The main difference between fine-tuning and RAG is that fine-tuning adjusts a model's parameters for specific tasks using additional training data, while RAG enhances a model's output by retrieving relevant information from an external source to inform its responses. Fine-tuning focuses on improving task performance, whereas RAG integrates external knowledge to generate more accurate and contextually relevant answers.


## 7) Evaluation: `Example`s + a metric

Optimizers need:
1. a small labeled `trainset` of `dspy.Example`s
2. a `metric(example, pred, trace=None)` that returns bool or float

In [26]:
class ShortAnswer(dspy.Signature):
    """Answer the question with a short factual phrase."""

    question: str = dspy.InputField()
    answer: str = dspy.OutputField(desc="Short phrase, not a full paragraph")


program = dspy.Predict(ShortAnswer)

trainset = [
    dspy.Example(question="Capital of France?", answer="Paris").with_inputs("question"),
    dspy.Example(question="2 + 2?", answer="4").with_inputs("question"),
    dspy.Example(question="Who wrote Hamlet?", answer="William Shakespeare").with_inputs("question"),
    dspy.Example(question="Chemical symbol for water?", answer="H2O").with_inputs("question"),
    dspy.Example(question="Largest planet in our solar system?", answer="Jupiter").with_inputs("question"),
    dspy.Example(question="Year the iPhone launched?", answer="2007").with_inputs("question"),
]

devset = [
    dspy.Example(question="Capital of Japan?", answer="Tokyo").with_inputs("question"),
    dspy.Example(question="Speed of light roughly in km/s?", answer="300000").with_inputs("question"),
    dspy.Example(question="Author of 1984?", answer="George Orwell").with_inputs("question"),
]


def normalize(text: str) -> str:
    return "".join(ch.lower() for ch in text if ch.isalnum())


def answer_contains(example, pred, trace=None) -> bool:
    """Pass if gold answer appears in the prediction (lenient for demos)."""
    gold = normalize(example.answer)
    got = normalize(pred.answer)
    return bool(gold) and gold in got


evaluate = dspy.Evaluate(
    devset=devset,
    metric=answer_contains,
    num_threads=4,
    display_progress=True,
    display_table=False,
)

baseline = evaluate(program)
print("baseline score:", baseline)

Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:00<00:00, 3473.06it/s] 

2026/08/10 22:37:56 INFO dspy.evaluate.evaluate: Average Metric: 2 / 3 (66.7%)



baseline score: 66.67


## 8) Optimize with `BootstrapFewShot`

The optimizer runs your program on the trainset, keeps traces that pass the metric, and injects those as few-shot demos.

This cell makes multiple LM calls — expect ~30–90 seconds.

In [27]:
optimizer = dspy.BootstrapFewShot(
    metric=answer_contains,
    max_bootstrapped_demos=3,
    max_labeled_demos=3,
    max_rounds=1,
)

optimized = optimizer.compile(student=program, trainset=trainset)
optimized_result = evaluate(optimized)

print("baseline :", baseline)
print("optimized:", optimized_result)

for ex in devset:
    pred = optimized(question=ex.question)
    print(f"Q: {ex.question}")
    print(f"  gold={ex.answer!r}  pred={pred.answer!r}  ok={answer_contains(ex, pred)}")

 50%|█████     | 3/6 [00:00<00:00, 1301.50it/s]


Bootstrapped 3 full traces after 3 examples for up to 1 rounds, amounting to 3 attempts.
Average Metric: 2.00 / 3 (66.7%): 100%|██████████| 3/3 [00:00<00:00, 4878.99it/s]

2026/08/10 22:37:56 INFO dspy.evaluate.evaluate: Average Metric: 2 / 3 (66.7%)



baseline : 66.67
optimized: 66.67
Q: Capital of Japan?
  gold='Tokyo'  pred='Tokyo'  ok=True
Q: Speed of light roughly in km/s?
  gold='300000'  pred='299,792 km/s'  ok=False
Q: Author of 1984?
  gold='George Orwell'  pred='George Orwell'  ok=True


In [28]:
# Inspect one optimized prompt (demos should appear)
_ = optimized(question="Capital of Italy?")
dspy.inspect_history(n=1)





[2026-08-10T22:37:56.786765]

System message:

Your input fields are:
1. `question` (str):
Your output fields are:
1. `answer` (str): Short phrase, not a full paragraph
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## question ## ]]
{question}

[[ ## answer ## ]]
{answer}

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Answer the question with a short factual phrase.


User message:

[[ ## question ## ]]
Capital of France?


Assistant message:

[[ ## answer ## ]]
Paris


User message:

[[ ## question ## ]]
2 + 2?


Assistant message:

[[ ## answer ## ]]
4


User message:

[[ ## question ## ]]
Who wrote Hamlet?


Assistant message:

[[ ## answer ## ]]
William Shakespeare


User message:

[[ ## question ## ]]
Capital of Italy?

Respond with the corresponding output fields, starting with the field `[[ ## answer ## ]]`, and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ##

## 9) Save / load the compiled program

In [29]:
path = "short_answer_optimized.json"
optimized.save(path)

reloaded = dspy.Predict(ShortAnswer)
reloaded.load(path)

print(reloaded(question="Capital of Italy?").answer)

Rome


## 10) (Optional) Tool-using agent with `ReAct`

Pass plain Python functions as tools. The LM decides when to call them.

In [30]:
def word_count(text: str) -> int:
    """Count whitespace-separated words in text."""
    return len(text.split())


def acronym(text: str) -> str:
    """Build an acronym from the first letter of each word."""
    return "".join(w[0].upper() for w in text.split() if w)


class TextTools(dspy.Signature):
    """Use tools when helpful, then answer."""

    request: str = dspy.InputField()
    result: str = dspy.OutputField()


agent = dspy.ReAct(TextTools, tools=[word_count, acronym])
print(agent(request="How many words are in 'retrieval augmented generation', and what is its acronym?").result)

The phrase 'retrieval augmented generation' has 3 words, and its acronym is RAG.


## Interview cheat sheet

| Concept | Role |
|---|---|
| **Signature** | Declare task I/O (`question -> answer`), not a prompt template |
| **Module** | How to execute the signature (`Predict`, `ChainOfThought`, `ReAct`, custom) |
| **Adapter** | Formats prompts / parses structured outputs under the hood |
| **Example** | Labeled datapoint; `.with_inputs(...)` marks which fields are inputs |
| **Metric** | `(example, pred, trace=None) -> bool \| float` used for eval + bootstrap |
| **Optimizer** | Compiles better demos/instructions against the metric |
| **BootstrapFewShot** | Cheap starter optimizer: keep successful traces as few-shot demos |
| **MIPROv2 / GEPA** | Heavier optimizers that also rewrite instructions (more LM calls) |

### Mental model
```
Signature  →  Module  →  Evaluate(metric)  →  Optimizer.compile  →  save()
```
